# Spanwise Average & Std — Filtered Temperature Windows
### Run cells in order: 1 → 2 → 3
### Reads filtered CSVs, computes column-wise mean and std across rows,
### saves summary CSV + PNG under `.../half_domain_temperature/summary_avg_std/`

In [1]:
# ══════════════════════════════════════════════════════════════════
# CELL 1 — CONFIGURATION
# ══════════════════════════════════════════════════════════════════

ROOT_DIR      = r"D:\FCAI\plain_coupon\infared_data"
DATA_SUBPATH  = r"time_average\average_temperature\half_domain_temperature"
FILTER_SUBDIR = "filtered_temp_average"
SUMMARY_SUBDIR= "summary_avg_std"

N_WORKERS = -1   # -1 = all cores; 1 = serial debug

WINDOWS = [
    {
        "input_csv"  : "window_1_filtered.csv",
        "output_csv" : "window_1_spanwise_avg_std.csv",
        "png_result" : "window_1_spanwise_avg_std.png",
        "label"      : "Window 1",
    },
    {
        "input_csv"  : "window_2_filtered.csv",
        "output_csv" : "window_2_spanwise_avg_std.csv",
        "png_result" : "window_2_spanwise_avg_std.png",
        "label"      : "Window 2",
    },
]

print("Config loaded.")
for w in WINDOWS:
    print(f"  {w['input_csv']}  →  {SUMMARY_SUBDIR}/{w['output_csv']}")
print(f"  Root : {ROOT_DIR}")

Config loaded.
  window_1_filtered.csv  →  summary_avg_std/window_1_spanwise_avg_std.csv
  window_2_filtered.csv  →  summary_avg_std/window_2_spanwise_avg_std.csv
  Root : D:\FCAI\plain_coupon\infared_data


In [2]:
# ══════════════════════════════════════════════════════════════════
# CELL 2 — IMPORTS & HELPERS
# ══════════════════════════════════════════════════════════════════
import os
import numpy as np
import pandas as pd
from pathlib import Path
from joblib import Parallel, delayed

def load_csv(csv_path):
    raw        = Path(csv_path).read_text(encoding="utf-8", errors="replace")
    lines      = [l.strip() for l in raw.splitlines() if l.strip()]
    col_counts = [len(l.split(",")) for l in lines]
    max_cols   = max(col_counts)
    data_lines = [l for l, c in zip(lines, col_counts) if c >= max_cols * 0.5]
    return np.array(
        [[float(v) for v in row.split(",")] for row in data_lines],
        dtype=np.float64
    )

def discover_cases(root_dir, data_subpath, input_csv):
    return sorted(
        [d for d in Path(root_dir).iterdir()
         if d.is_dir() and (d / data_subpath / input_csv).exists()]
    )

print("Helpers ready.")

Helpers ready.


In [3]:
# ══════════════════════════════════════════════════════════════════
# CELL 3 — PARALLEL SPANWISE AVERAGE & STD
#
# For each filtered CSV:
#   - average  across rows → function of column index
#   - std       across rows → function of column index
#
# Output CSV columns:
#   col_index | spanwise_avg [K] | spanwise_std [K]
#
# Output PNG:
#   top panel    — spanwise average with ±1σ shaded band
#   bottom panel — spanwise std
# ══════════════════════════════════════════════════════════════════
%matplotlib inline

# ── Worker ───────────────────────────────────────────────────────
def _spanwise_one(job):
    import numpy as np
    import pandas as pd
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from pathlib import Path
    import traceback

    name       = job["name"]
    win_label  = job["win_label"]
    csv_in     = Path(job["csv_in"])
    out_dir    = Path(job["out_dir"])
    out_dir.mkdir(parents=True, exist_ok=True)
    output_csv = out_dir / job["output_csv"]
    output_png = out_dir / job["png_result"]

    log = []

    def _load(path):
        raw        = path.read_text(encoding="utf-8", errors="replace")
        lines      = [l.strip() for l in raw.splitlines() if l.strip()]
        col_counts = [len(l.split(",")) for l in lines]
        max_cols   = max(col_counts)
        data_lines = [l for l, c in zip(lines, col_counts) if c >= max_cols * 0.5]
        return np.array(
            [[float(v) for v in row.split(",")] for row in data_lines],
            dtype=np.float64
        )

    try:
        data = _load(csv_in)
        n_rows, n_cols = data.shape
        log.append(f"    shape={data.shape}  T=[{data.min():.2f}, {data.max():.2f}] °C")

        # ── Spanwise statistics (average over row direction) ──────
        col_idx  = np.arange(n_cols)          # 0, 1, 2, ..., n_cols-1
        avg      = data.mean(axis=0)           # shape (n_cols,)
        std      = data.std(axis=0, ddof=1)    # unbiased std, shape (n_cols,)

        log.append(f"    spanwise avg: min={avg.min():.3f} °C  max={avg.max():.3f} °C")
        log.append(f"    spanwise std: min={std.min():.4f} °C  max={std.max():.4f} °C")

        # ── Save CSV ──────────────────────────────────────────────
        df = pd.DataFrame({
            "col_index"      : col_idx,
            "spanwise_avg_C" : avg,
            "spanwise_std_C" : std,
        })
        df.to_csv(output_csv, index=False)
        log.append(f"    CSV  → {output_csv}")

        # ── Save PNG ──────────────────────────────────────────────
        fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

        # Top: average + ±1σ shaded band
        axes[0].plot(col_idx, avg, color="#C0392B", linewidth=1.2, label="Spanwise avg")
        axes[0].fill_between(col_idx, avg - std, avg + std,
                             alpha=0.25, color="#C0392B", label="±1σ band")
        axes[0].set_ylabel("Temperature [°C]", fontsize=10)
        axes[0].set_title(f"{name} — {win_label}\nSpanwise average (rows averaged)",
                          fontsize=10)
        axes[0].legend(fontsize=9)
        axes[0].grid(True, linestyle="--", alpha=0.4)

        # Bottom: std only
        axes[1].plot(col_idx, std, color="#2980B9", linewidth=1.2)
        axes[1].fill_between(col_idx, 0, std, alpha=0.2, color="#2980B9")
        axes[1].set_ylabel("Std deviation [°C]", fontsize=10)
        axes[1].set_xlabel("Column index", fontsize=10)
        axes[1].set_title("Spanwise standard deviation", fontsize=10)
        axes[1].grid(True, linestyle="--", alpha=0.4)

        plt.tight_layout()
        plt.savefig(output_png, dpi=150, bbox_inches="tight")
        plt.close(fig)
        log.append(f"    PNG  → {output_png}")

        return (name, win_label, "OK", "\n".join(log))

    except Exception as e:
        log.append(f"    ERROR: {e}")
        log.append(traceback.format_exc())
        return (name, win_label, "FAILED", "\n".join(log))


# ── Build job list ────────────────────────────────────────────────
jobs = []
for win_cfg in WINDOWS:
    cases = discover_cases(
        ROOT_DIR,
        str(Path(DATA_SUBPATH) / FILTER_SUBDIR),
        win_cfg["input_csv"]
    )
    print(f"  {win_cfg['label']}: {len(cases)} cases found")

    for case_dir in cases:
        jobs.append({
            "name"      : case_dir.name,
            "win_label" : win_cfg["label"],
            "csv_in"    : str(case_dir / DATA_SUBPATH / FILTER_SUBDIR / win_cfg["input_csv"]),
            "out_dir"   : str(case_dir / DATA_SUBPATH / SUMMARY_SUBDIR),
            "output_csv": win_cfg["output_csv"],
            "png_result": win_cfg["png_result"],
        })

n_workers = N_WORKERS if N_WORKERS != -1 else os.cpu_count()
print(f"\nDispatching {len(jobs)} jobs across {n_workers} workers...\n")

# ── Run in parallel ───────────────────────────────────────────────
results = Parallel(n_jobs=N_WORKERS, backend="loky", verbose=10)(
    delayed(_spanwise_one)(job) for job in jobs
)

# ── Print logs & summary ──────────────────────────────────────────
grand_log = []
for name, win_label, status, logs in results:
    grand_log.append((name, win_label, status))
    mark = "✓" if status == "OK" else "✗"
    print(f"\n{mark} [{name}] {win_label}")
    print(logs)

ok     = [r for r in grand_log if r[2] == "OK"]
failed = [r for r in grand_log if r[2] == "FAILED"]
print("\n" + "=" * 62)
print(f"ALL DONE — {len(ok)}/{len(grand_log)} succeeded"
      + (f",  {len(failed)} failed" if failed else ""))
if failed:
    print("\nFailed jobs:")
    for name, win, status in grand_log:
        if status == "FAILED":
            print(f"  ✗  {name}  /  {win}")
print("=" * 62)


  Window 1: 15 cases found
  Window 2: 15 cases found

Dispatching 30 jobs across 256 workers...



[Parallel(n_jobs=-1)]: Using backend LokyBackend with 61 concurrent workers.
[Parallel(n_jobs=-1)]: Done   5 out of  30 | elapsed:    2.1s remaining:   11.1s
[Parallel(n_jobs=-1)]: Done   9 out of  30 | elapsed:    2.2s remaining:    5.2s



✓ [cr_1400_phi_085] Window 1
    shape=(75, 150)  T=[206.56, 447.33] °C
    spanwise avg: min=222.262 °C  max=439.051 °C
    spanwise std: min=7.2320 °C  max=33.3796 °C
    CSV  → D:\FCAI\plain_coupon\infared_data\cr_1400_phi_085\time_average\average_temperature\half_domain_temperature\summary_avg_std\window_1_spanwise_avg_std.csv
    PNG  → D:\FCAI\plain_coupon\infared_data\cr_1400_phi_085\time_average\average_temperature\half_domain_temperature\summary_avg_std\window_1_spanwise_avg_std.png

✓ [cr_1400_phi_09] Window 1
    shape=(75, 150)  T=[207.76, 488.16] °C
    spanwise avg: min=223.824 °C  max=480.171 °C
    spanwise std: min=7.1664 °C  max=37.7639 °C
    CSV  → D:\FCAI\plain_coupon\infared_data\cr_1400_phi_09\time_average\average_temperature\half_domain_temperature\summary_avg_std\window_1_spanwise_avg_std.csv
    PNG  → D:\FCAI\plain_coupon\infared_data\cr_1400_phi_09\time_average\average_temperature\half_domain_temperature\summary_avg_std\window_1_spanwise_avg_std.png

✓ [cr_

[Parallel(n_jobs=-1)]: Done  13 out of  30 | elapsed:    2.4s remaining:    3.2s
[Parallel(n_jobs=-1)]: Done  17 out of  30 | elapsed:    2.4s remaining:    1.8s
[Parallel(n_jobs=-1)]: Done  21 out of  30 | elapsed:    2.4s remaining:    1.0s
[Parallel(n_jobs=-1)]: Done  25 out of  30 | elapsed:    2.4s remaining:    0.4s
[Parallel(n_jobs=-1)]: Done  30 out of  30 | elapsed:    2.4s finished
